### Name  Utkarsh Pateriya 
### Roll No  SCFP125048
### Lab 7 — Agentic RAG On Union Budget

In [14]:
!pip install -q langchain-core langchain-text-splitters langchain-huggingface langchain-chroma chromadb "langchain-docling[local]" langsmith langchain-groq langchain-community pypdf pandas

In [15]:
import pandas as pd
from kaggle_secrets import UserSecretsClient
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, ToolMessage

In [ ]:
docs = PyPDFLoader("/kaggle/input/datasets/utkarshpateriya/budget/budget_speech.pdf").load()
for d in docs:
    d.metadata["page"] = d.metadata.get("page", 0) + 1
    d.page_content = " ".join(d.page_content.replace("-\n", "").split())  # Fix broken words & extra spaces

texts = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150).split_documents(docs)

def clean_metadata(doc: Document) -> Document:
    simple = {k: v for k, v in doc.metadata.items() if isinstance(v, (str, int, float, bool))}
    return Document(page_content=doc.page_content, metadata=simple)

clean_texts = [clean_metadata(d) for d in texts]
print(f"Q1 Chunks: {len(clean_texts)} | Metadata: {clean_texts[0].metadata}")

Q1 Chunks: 172 | Metadata: {'producer': 'doPDF Ver 8.4 Build 935', 'creator': 'Adobe Acrobat (64-bit) 25.1.20997', 'creationdate': '2026-02-01T05:37:46+05:30', 'moddate': '2026-02-01T05:39:37+05:30', 'title': '', 'source': '/kaggle/input/datasets/mohamedzaidshaikh/budget/budget_speech.pdf', 'total_pages': 65, 'page': 1, 'page_label': '1'}


In [ ]:
emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
Chroma.from_documents(clean_texts, emb, persist_directory="./db")
db = Chroma(persist_directory="./db", embedding_function=emb)

print("\nQ2 Top 3 Results:")
for doc, score in db.similarity_search_with_score("Biopharma SHAKTI outlay", k=3):
    print(f"Score: {score:.4f} | Page {doc.metadata['page']}: {doc.page_content[:120]}...")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Q2 Top 3 Results:
Score: 0.6370 | Page 8: 4 Scaling up manufacturing in 7 strategic and frontier sectors: 16. Biopharma SHAKTI (Strategy for Healthcare Advancemen...
Score: 0.6370 | Page 8: 4 Scaling up manufacturing in 7 strategic and frontier sectors: 16. Biopharma SHAKTI (Strategy for Healthcare Advancemen...
Score: 0.6370 | Page 8: 4 Scaling up manufacturing in 7 strategic and frontier sectors: 16. Biopharma SHAKTI (Strategy for Healthcare Advancemen...


In [ ]:
@tool
def search_budget(query: str) -> str:
    """Search the Budget 2026-27 speech for schemes, outlays, tax rates, and fiscal numbers."""
    return "\n".join(f"[Page {d.metadata['page']}] {d.page_content}" for d in db.similarity_search(query, k=4))

@tool
def calculator(expr: str) -> str:
    """Calculate math expressions like '(12.2 - 11.2) / 11.2 * 100'."""
    return str(round(eval(expr, {"__builtins__": {}}), 2))

tools = {"search_budget": search_budget, "calculator": calculator}
llm = ChatGroq(model="openai/gpt-oss-120b", api_key=UserSecretsClient().get_secret("GROQ_API_KEY"), temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer greetings/general concepts directly without tools. Use `search_budget` for budget facts and cite [Page X]. "
               "If not in the document, say so. Use `calculator` for math."),
    MessagesPlaceholder("history")
])
chain = prompt | llm.bind_tools(list(tools.values()))

In [ ]:
queries = [
    "What is the outlay of Biopharma SHAKTI?",
    "How does the 2026-27 fiscal deficit compare with 2025-26?",
    "What changes for NCCD on chewing tobacco, and what is the effective duty?",
    "Hi, how are you?",
    "Explain fiscal deficit in simple words.",
    "What are the new income tax slab rates in this budget?",
    "By what percentage does public capex rise from BE 2025-26 to FY2026-27?",
    "What is the combined outlay of Biopharma SHAKTI, the Container Manufacturing scheme and CCUS?",
    "exit"
]

q_iter, rows = iter(queries), []
while True:
    q = next(q_iter)  # Use input("User: ") for manual typing
    if q.lower() == "exit":
        break
    
    history, used = [HumanMessage(content=q)], []
    for _ in range(5):  # Max 5 tool rounds
        ai_msg = chain.invoke({"history": history})
        history.append(ai_msg)
        if not ai_msg.tool_calls:
            break
        for tc in ai_msg.tool_calls:
            used.append(tc["name"])
            history.append(ToolMessage(content=tools[tc["name"]].invoke(tc["args"]), tool_call_id=tc["id"]))
            
    ans = history[-1].content
    print(f"\nQ: {q}\nTools: {used}\nA: {ans}")
    rows.append({"Query": q, "Searched (Y/N)": "Y" if "search_budget" in used else "N", "Tools Used": ", ".join(used) or "None", "Final Answer": ans})

display(pd.DataFrame(rows))
print("\nDecision Accuracy: 8/8 (100%) - Searched only for budget-specific questions (Q1-3, Q6-8) and skipped for Q4-5.")


Q: What is the outlay of Biopharma SHAKTI?
Tools: ['search_budget']
A: The Biopharma SHAKTI programme is proposed with an outlay of **₹ 10,000 crores** over the next five years【Page 8】.

Q: How does the 2026-27 fiscal deficit compare with 2025-26?
Tools: ['search_budget', 'search_budget', 'search_budget']
A: The Budget 2026‑27 estimates a fiscal deficit of **4.3 % of GDP**.  
In the Revised Estimates for FY 2025‑26 the fiscal deficit was **4.4 % of GDP**【Page 23】.

**Comparison**

* The deficit falls by **0.1 percentage‑point** (from 4.4 % to 4.3 % of GDP).  
* In relative terms this is a reduction of about **2.3 %** ((4.3 – 4.4)/4.4 × 100)【calculator】.

So, the 2026‑27 fiscal deficit is slightly lower than the 2025‑26 deficit, reflecting a modest improvement in fiscal consolidation.

Q: What changes for NCCD on chewing tobacco, and what is the effective duty?
Tools: ['search_budget']
A: The budget proposes to raise the **National Calamity Contingent Duty (NCCD) rate** on chewing toba

,Query,Searched (Y/N),Tools Used,Final Answer
0,What is the outlay of Biopharma SHAKTI?,Y,search_budget,The Biopharma SHAKTI programme is proposed wit...
1,How does the 2026-27 fiscal deficit compare wi...,Y,"search_budget, search_budget, search_budget",The Budget 2026‑27 estimates a fiscal deficit ...
2,"What changes for NCCD on chewing tobacco, and ...",Y,search_budget,The budget proposes to raise the **National Ca...
3,"Hi, how are you?",N,None,"I'm doing great, thank you! How can I help you..."
4,Explain fiscal deficit in simple words.,N,None,A fiscal deficit happens when a government spe...
5,What are the new income tax slab rates in this...,Y,"search_budget, search_budget, search_budget, s...","[Page 6] Dialogue 2026, several innovative ide..."
6,By what percentage does public capex rise from...,Y,"search_budget, search_budget, search_budget, s...",[Page 23] 19 estimated at par with BE of 2025 ...
7,What is the combined outlay of Biopharma SHAKT...,Y,"search_budget, search_budget, search_budget",The Budget proposes the following outlays:\n\n...



Decision Accuracy: 8/8 (100%) - Searched only for budget-specific questions (Q1-3, Q6-8) and skipped for Q4-5.
